# Week 8 Exercise — The 4-Model Deal Hunter

An upgrade to the course capstone that wires together everything from Weeks 5–8:

- **Claude Sonnet** replaces GPT-5.1 as the frontier pricing model  
- **RAG Few-Shot** (our Week 7 technique) joins as a 4th ensemble member  
- **Notebook-native output** replaces Pushover — every deal gets a Claude-crafted alert  

### Architecture

```
RSS feeds ──► CustomScannerAgent (Groq llama-3.3-70b JSON mode)
                          │
                  5 curated deals
                          │
     ┌────────────────────┼────────────────────────┐
     ▼                    ▼                         ▼
ClaudeFrontierAgent  SpecialistAgent         RAGFewShotAgent   NeuralNetworkAgent
(Claude+Chroma RAG)  (Modal/QLoRA Llama)    (k-NN+Groq)       (DNN, optional)
   weight=0.50          weight=0.20            weight=0.20        weight=0.10
     └────────────────────┼────────────────────────┘
                          ▼
                  CustomEnsembleAgent
            (weighted average; auto-renorm if DNN absent)
                          ▼
              Rank deals by (estimate − listed price)
                          ▼
              Best deal + Claude-crafted alert printed in notebook
```


In [1]:
import sys
import os
import re
from pathlib import Path
from typing import Optional, List, Dict, Tuple

import numpy as np
from dotenv import load_dotenv
from IPython.display import display, Markdown, HTML


In [2]:
def _repo_root() -> Path:
    """Walk up from cwd to find the repo root (the dir containing pyproject.toml)."""
    p = Path.cwd()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    return p

REPO   = _repo_root()
WEEK8  = REPO / "week8"

# week8/ on sys.path so `from agents.xxx import Xxx` works
for d in [str(REPO), str(WEEK8)]:
    if d not in sys.path:
        sys.path.insert(0, d)

load_dotenv(REPO / ".env", override=True)

REQUIRED = {
    "OPENAI_API_KEY":    "CustomScannerAgent (gpt-4o-mini)",
    "ANTHROPIC_API_KEY": "ClaudeFrontierAgent + MessagingAgent",
    "GROQ_API_KEY":      "RAGFewShotAgent + Preprocessor",
    "HF_TOKEN":          "items_lite dataset download",
}
missing = [k for k, _ in REQUIRED.items() if not os.getenv(k)]
if missing:
    print("WARNING — missing keys (some agents may not work):", missing)
else:
    print("All required API keys present.")

# Point Preprocessor at Groq so we don't need a local Ollama instance
os.environ.setdefault("PRICER_PREPROCESSOR_MODEL", "groq/llama-3.1-8b-instant")


All required API keys present.


'groq/llama-3.1-8b-instant'

In [3]:
import chromadb
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

CHROMA_PATH = WEEK8 / "chroma_exercise"
COLLECTION_NAME = "items_lite_train"
MAX_ITEMS = 5_000   # keep build time reasonable; FrontierAgent finds good enough 5-NN

def build_or_load_chroma() -> chromadb.Collection:
    """Return a Chroma collection of training items, building it if necessary."""
    client = chromadb.PersistentClient(path=str(CHROMA_PATH))
    collection = client.get_or_create_collection(COLLECTION_NAME)

    if collection.count() >= MAX_ITEMS:
        print(f"Chroma collection already contains {collection.count()} items — skipping build.")
        return collection

    print(f"Building Chroma collection from ed-donner/items_lite (up to {MAX_ITEMS} rows)…")
    ds = load_dataset("ed-donner/items_lite", split=f"train[:{MAX_ITEMS}]")
    embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    docs, metas, ids = [], [], []
    for i, row in enumerate(ds):
        text = row.get("full") or row.get("summary") or row.get("title", "")
        if not text:
            continue
        docs.append(text[:512])
        metas.append({"price": float(row["price"])})
        ids.append(str(i))

    # Embed in batches of 256 for memory efficiency
    BATCH = 256
    all_vecs = []
    for start in tqdm(range(0, len(docs), BATCH), desc="Embedding"):
        batch = docs[start : start + BATCH]
        all_vecs.append(embedder.encode(batch, show_progress_bar=False))
    all_vecs = np.vstack(all_vecs)

    # Upsert in batches
    for start in range(0, len(docs), BATCH):
        collection.upsert(
            documents=docs[start : start + BATCH],
            embeddings=all_vecs[start : start + BATCH].astype(float).tolist(),
            metadatas=metas[start : start + BATCH],
            ids=ids[start : start + BATCH],
        )
    print(f"Chroma collection ready: {collection.count()} items.")
    return collection

collection = build_or_load_chroma()


Chroma collection already contains 5000 items — skipping build.


In [4]:
import anthropic
from openai import OpenAI
from groq import Groq as GroqClient
import modal

from agents.agent import Agent
from agents.deals import ScrapedDeal, DealSelection, Deal, Opportunity
from agents.deep_neural_network import DeepNeuralNetworkInference
from agents.preprocessor import Preprocessor
from litellm import completion as litellm_completion


# ─────────────────────────────────────────────────────────────────────────────
# CustomScannerAgent — uses Groq (free tier) with JSON mode instead of OpenAI
# ─────────────────────────────────────────────────────────────────────────────
class CustomScannerAgent(Agent):
    """Scans RSS feeds and uses Groq llama-3.3-70b with JSON mode to pick top 5 deals."""
    name = "Scanner Agent"
    color = Agent.CYAN
    GROQ_MODEL = "llama-3.3-70b-versatile"

    SYSTEM_PROMPT = (
        "You identify and summarize the 5 most detailed deals from a list. "
        "Select deals that have the most detailed, high-quality description and the most clear price. "
        'Respond ONLY with a JSON object matching this exact schema: '
        '{"deals": [{"product_description": "...", "price": 123.0, "url": "..."}, ...]}. '
        "If the price of a deal isn't clear, do not include it. "
        "Be careful with '$XXX off' deals — that is not the actual price."
    )
    USER_PROMPT_PREFIX = (
        "Respond with the most promising 5 deals from this list. "
        "Rephrase each description to summarize the product itself (not the deal terms). "
        "Write a short paragraph in the product_description field for each item.\n\nDeals:\n\n"
    )
    USER_PROMPT_SUFFIX = "\n\nReturn exactly 5 deals as a JSON object with a 'deals' array."

    def __init__(self, groq_client: GroqClient):
        self.log("Custom Scanner Agent initializing with Groq llama-3.3-70b-versatile")
        self.groq = groq_client
        self.log("Custom Scanner Agent ready")

    def fetch_deals(self, memory: List[Opportunity]) -> List[ScrapedDeal]:
        self.log("Fetching deals from RSS feeds")
        seen_urls = {opp.deal.url for opp in memory}
        scraped = ScrapedDeal.fetch()
        result = [s for s in scraped if s.url not in seen_urls]
        self.log(f"Fetched {len(result)} new deals")
        return result

    def make_user_prompt(self, scraped: List[ScrapedDeal]) -> str:
        return self.USER_PROMPT_PREFIX + "\n\n".join(s.describe() for s in scraped) + self.USER_PROMPT_SUFFIX

    def scan(self, memory: List[Opportunity] = []) -> Optional[DealSelection]:
        import json
        scraped = self.fetch_deals(memory)
        if not scraped:
            return None
        user_prompt = self.make_user_prompt(scraped)
        self.log(f"Calling {self.GROQ_MODEL} via Groq with JSON mode")
        try:
            resp = self.groq.chat.completions.create(
                model=self.GROQ_MODEL,
                messages=[
                    {"role": "system", "content": self.SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                response_format={"type": "json_object"},
                max_tokens=2048,
            )
            data = json.loads(resp.choices[0].message.content)
            selection = DealSelection(**data)
            selection.deals = [d for d in selection.deals if d.price > 0]
            self.log(f"Scanner selected {len(selection.deals)} deals")
            return selection
        except Exception as e:
            self.log(f"Scanner failed: {e}")
            return None

    def test_scan(self, memory: List[Opportunity] = []) -> DealSelection:
        """Return hardcoded test deals for offline development."""
        return DealSelection(**{
            "deals": [
                {"product_description": "Hisense 55-inch 4K UHD Roku Smart TV with Dolby Vision HDR, 3 HDMI ports, and built-in Alexa/Google Assistant voice control.", "price": 178, "url": "https://www.dealnews.com/products/Hisense/55-4K-Roku-TV/1.html"},
                {"product_description": "Lenovo IdeaPad Slim 5 laptop with AMD Ryzen 5, 16-inch 1080p touch display, 16 GB RAM, 512 GB SSD.", "price": 446, "url": "https://www.dealnews.com/products/Lenovo/IdeaPad-Slim-5/2.html"},
                {"product_description": "Dell G15 gaming laptop with AMD Ryzen 5, 15.6-inch 120Hz display, Nvidia RTX 3050, 16 GB RAM, 1 TB NVMe SSD.", "price": 650, "url": "https://www.dealnews.com/products/Dell/G15-Gaming/3.html"},
                {"product_description": "Anker 10,000 mAh portable charger with 20W USB-C Power Delivery and dual USB-A outputs.", "price": 22, "url": "https://www.dealnews.com/products/Anker/PowerCore-10K/4.html"},
                {"product_description": "Samsung 65-inch QLED 4K smart TV with Quantum Processor, 120Hz panel, and 4 HDMI 2.1 ports.", "price": 798, "url": "https://www.dealnews.com/products/Samsung/Q70C-QLED/5.html"},
            ]
        })


# ─────────────────────────────────────────────────────────────────────────────
# ClaudeFrontierAgent — Chroma RAG + Claude Sonnet
# ─────────────────────────────────────────────────────────────────────────────
class ClaudeFrontierAgent(Agent):
    """Uses Chroma to find 5 similar items then asks Claude to price the product."""
    name = "Claude Frontier Agent"
    color = Agent.BLUE
    MODEL = "claude-sonnet-4-5"

    def __init__(self, chroma_collection):
        self.log("Initializing Claude Frontier Agent")
        self.client = anthropic.Anthropic()
        self.collection = chroma_collection
        self.embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        self.log("Claude Frontier Agent ready")

    def _find_similars(self, description: str) -> Tuple[List[str], List[float]]:
        vec = self.embedder.encode([description]).astype(float).tolist()
        results = self.collection.query(query_embeddings=vec, n_results=5)
        docs = results["documents"][0]
        prices = [m["price"] for m in results["metadatas"][0]]
        return docs, prices

    def _build_prompt(self, description: str, docs: List[str], prices: List[float]) -> str:
        ctx = "Here are some similar products for price context:\n\n"
        for d, p in zip(docs, prices):
            ctx += f"Product: {d[:200]}\nPrice: ${p:.2f}\n\n"
        return (
            f"Estimate the price of this product. Respond with the price only (a number), no explanation.\n\n"
            f"{description}\n\n{ctx}"
        )

    @staticmethod
    def _extract_price(text: str) -> float:
        text = text.replace("$", "").replace(",", "")
        m = re.search(r"[-+]?\d*\.\d+|\d+", text)
        return float(m.group()) if m else 0.0

    def price(self, description: str) -> float:
        docs, prices = self._find_similars(description)
        prompt = self._build_prompt(description, docs, prices)
        self.log(f"Calling {self.MODEL}")
        try:
            resp = self.client.messages.create(
                model=self.MODEL,
                max_tokens=64,
                messages=[{"role": "user", "content": prompt}],
            )
            result = self._extract_price(resp.content[0].text)
        except Exception as e:
            self.log(f"Claude Frontier error: {e}")
            result = 0.0
        self.log(f"Claude Frontier predicting ${result:.2f}")
        return result


# ─────────────────────────────────────────────────────────────────────────────
# RAGFewShotAgent — in-memory cosine similarity k-NN + Groq
# (our Week 7 technique reused here as a 4th ensemble member)
# ─────────────────────────────────────────────────────────────────────────────
class RAGFewShotAgent(Agent):
    """Retrieves k similar training items via cosine similarity, injects them as
    few-shot examples into a Groq prompt — no fine-tuning required."""
    name = "RAG Few-Shot Agent"
    color = Agent.GREEN
    GROQ_MODEL = "llama-3.3-70b-versatile"
    K = 5

    def __init__(self, train_rows: List[Dict], groq_client: GroqClient):
        self.log("Initializing RAG Few-Shot Agent")
        self.groq = groq_client
        self.embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

        descriptions = []
        self.prices: List[float] = []
        for row in train_rows:
            text = row.get("full") or row.get("summary") or row.get("title", "")
            descriptions.append(text[:512])
            self.prices.append(float(row["price"]))

        self.index = self.embedder.encode(descriptions, show_progress_bar=True, batch_size=256)
        self.descriptions = descriptions
        self.log(f"RAG Few-Shot Agent indexed {len(descriptions)} training items")

    def _retrieve(self, query: str) -> List[Tuple[str, float]]:
        qvec = self.embedder.encode([query])
        sims = (self.index @ qvec.T).squeeze()
        top_k = sims.argsort()[::-1][: self.K]
        return [(self.descriptions[i], self.prices[i]) for i in top_k]

    def _build_messages(self, query: str, examples: List[Tuple[str, float]]) -> List[Dict]:
        msgs = []
        for desc, price in examples:
            msgs.append({"role": "user", "content": f"Price this product:\n{desc}"})
            msgs.append({"role": "assistant", "content": f"${price:.2f}"})
        msgs.append({"role": "user", "content": f"Price this product:\n{query}"})
        return msgs

    @staticmethod
    def _extract_price(text: str) -> float:
        text = text.replace("$", "").replace(",", "")
        m = re.search(r"[-+]?\d*\.\d+|\d+", text)
        return float(m.group()) if m else 0.0

    def price(self, description: str) -> float:
        examples = self._retrieve(description)
        msgs = self._build_messages(description, examples)
        self.log(f"Calling {self.GROQ_MODEL} via Groq with {self.K} few-shot examples")
        try:
            resp = self.groq.chat.completions.create(
                model=self.GROQ_MODEL, messages=msgs, max_tokens=64
            )
            result = self._extract_price(resp.choices[0].message.content)
        except Exception as e:
            self.log(f"RAG Few-Shot error: {e}")
            result = 0.0
        self.log(f"RAG Few-Shot predicting ${result:.2f}")
        return result


# ─────────────────────────────────────────────────────────────────────────────
# CustomSpecialistAgent — wraps the Modal fine-tuned Llama, falls back gracefully
# ─────────────────────────────────────────────────────────────────────────────
class CustomSpecialistAgent(Agent):
    """Calls the QLoRA fine-tuned Llama deployed on Modal.
    Handles both the @app.cls (Pricer class) and @app.function (price fn) form."""
    name = "Specialist Agent"
    color = Agent.RED

    def __init__(self):
        self.log("Specialist Agent connecting to Modal")
        self._pricer = None
        self._fn = None
        self._available = False
        try:
            Pricer = modal.Cls.from_name("pricer-service", "Pricer")
            self._pricer = Pricer()
            self._available = True
            self.log("Specialist Agent connected via Pricer class")
        except Exception:
            try:
                self._fn = modal.Function.from_name("pricer-service", "price")
                self._available = True
                self.log("Specialist Agent connected via function form")
            except Exception as e:
                self.log(f"Specialist Agent unavailable (Modal not reachable): {e}")

    def price(self, description: str) -> float:
        if not self._available:
            return 0.0
        self.log("Specialist Agent calling Modal")
        try:
            if self._pricer is not None:
                result = self._pricer.price.remote(description)
            else:
                result = self._fn.remote(description)
            self.log(f"Specialist Agent predicting ${result:.2f}")
            return result
        except Exception as e:
            self.log(f"Specialist Agent call failed: {e}")
            return 0.0


# ─────────────────────────────────────────────────────────────────────────────
# OptionalNeuralNetworkAgent — graceful no-op if weights are missing
# ─────────────────────────────────────────────────────────────────────────────
class OptionalNeuralNetworkAgent(Agent):
    """Wraps the DNN agent; silently disabled if deep_neural_network.pth is absent."""
    name = "DNN Agent"
    color = Agent.MAGENTA

    def __init__(self):
        weights = WEEK8 / "deep_neural_network.pth"
        self._available = False
        if not weights.exists():
            self.log(f"DNN weights not found at {weights} — agent disabled")
            return
        try:
            self.log("Loading DNN weights")
            self._nn = DeepNeuralNetworkInference()
            self._nn.setup()
            self._nn.load(str(weights))
            self._available = True
            self.log("DNN Agent ready")
        except Exception as e:
            self.log(f"DNN Agent failed to load: {e}")

    def price(self, description: str) -> float:
        if not self._available:
            return 0.0
        try:
            result = self._nn.inference(description)
            self.log(f"DNN predicting ${result:.2f}")
            return result
        except Exception as e:
            self.log(f"DNN inference error: {e}")
            return 0.0


# ─────────────────────────────────────────────────────────────────────────────
# CustomEnsembleAgent — weighted average, auto-renormalises if DNN is absent
# ─────────────────────────────────────────────────────────────────────────────
WEIGHTS = {"claude": 0.50, "specialist": 0.20, "rag": 0.20, "dnn": 0.10}

class CustomEnsembleAgent(Agent):
    """4-model weighted ensemble: Claude(0.5) + Specialist(0.2) + RAG(0.2) + DNN(0.1)."""
    name = "Ensemble Agent"
    color = Agent.YELLOW

    def __init__(
        self,
        claude_agent: ClaudeFrontierAgent,
        specialist_agent: CustomSpecialistAgent,
        rag_agent: RAGFewShotAgent,
        dnn_agent: OptionalNeuralNetworkAgent,
        preprocessor: Preprocessor,
    ):
        self.claude      = claude_agent
        self.specialist  = specialist_agent
        self.rag         = rag_agent
        self.dnn         = dnn_agent
        self.preprocessor = preprocessor
        self.log("Custom Ensemble Agent ready")

    def price(self, description: str) -> Dict:
        """Return dict of per-agent estimates and the weighted ensemble price."""
        self.log("Preprocessing description")
        try:
            rewrite = self.preprocessor.preprocess(description)
        except Exception as e:
            self.log(f"Preprocessor failed ({e}), using raw description")
            rewrite = description

        raw = {
            "claude":     self.claude.price(rewrite),
            "specialist": self.specialist.price(rewrite),
            "rag":        self.rag.price(rewrite),
            "dnn":        self.dnn.price(rewrite),
        }

        # Weighted average over agents that returned a positive estimate
        total_w, total_p = 0.0, 0.0
        for name, w in WEIGHTS.items():
            p = raw[name]
            if p > 0:
                total_p += p * w
                total_w += w

        ensemble = total_p / total_w if total_w > 0 else 0.0
        raw["ensemble"] = ensemble
        return raw


In [5]:
print("Loading training data for RAGFewShotAgent embedding index…")
train_ds = load_dataset("ed-donner/items_lite", split="train[:5000]")
train_rows = list(train_ds)

groq_client  = GroqClient(api_key=os.getenv("GROQ_API_KEY"))

print("Initializing agents…")
claude_agent     = ClaudeFrontierAgent(collection)
specialist_agent = CustomSpecialistAgent()
rag_agent        = RAGFewShotAgent(train_rows, groq_client)
dnn_agent        = OptionalNeuralNetworkAgent()
preprocessor     = Preprocessor()
ensemble_agent   = CustomEnsembleAgent(claude_agent, specialist_agent, rag_agent, dnn_agent, preprocessor)
scanner_agent    = CustomScannerAgent(groq_client)

print("All agents initialized.")


Loading training data for RAGFewShotAgent embedding index…
Initializing agents…


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

All agents initialized.


In [6]:
SAMPLE = (
    "Lenovo IdeaPad Slim 5 laptop, AMD Ryzen 5 8645HS, 16-inch 1080p touch display, "
    "16 GB RAM, 512 GB NVMe SSD, Windows 11."
)
print("Testing ensemble on a sample product…\n", SAMPLE[:80], "…")
sample_result = ensemble_agent.price(SAMPLE)
for agent_name, est in sample_result.items():
    label = f"{agent_name:12s}"
    bar   = ("$" + f"{est:.2f}") if est > 0 else "(unavailable)"
    print(f"  {label}: {bar}")


Testing ensemble on a sample product…
 Lenovo IdeaPad Slim 5 laptop, AMD Ryzen 5 8645HS, 16-inch 1080p touch display, 1 …


INFO:backoff:Backing off send_request(...) for 0.2s (requests.exceptions.ReadTimeout: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Read timed out. (read timeout=15))


  claude      : $725.00
  specialist  : $700.00
  rag         : $729.00
  dnn         : (unavailable)
  ensemble    : $720.33


In [7]:
# Set USE_TEST_SCAN = True to avoid hitting live RSS feeds (faster for debugging)
USE_TEST_SCAN = False

print("Scanning for deals…")
if USE_TEST_SCAN:
    selection = scanner_agent.test_scan()
else:
    selection = scanner_agent.scan()

if selection is None or not selection.deals:
    print("No deals found — switching to test scan.")
    selection = scanner_agent.test_scan()

print(f"\nScanner found {len(selection.deals)} deals. Pricing each with the ensemble…\n")

results: List[Dict] = []
for deal in selection.deals:
    print(f"  Pricing: {deal.product_description[:70]}…")
    estimates = ensemble_agent.price(deal.product_description)
    discount  = estimates["ensemble"] - deal.price
    results.append({
        "deal":      deal,
        "estimates": estimates,
        "discount":  discount,
    })
    print(f"    listed=${deal.price:.2f}  ensemble=${estimates['ensemble']:.2f}  discount=${discount:.2f}")

results.sort(key=lambda r: r["discount"], reverse=True)
print("\nDone pricing all deals.")


Scanning for deals…

Scanner found 5 deals. Pricing each with the ensemble…

  Pricing: The Apple Watch SE 2 40mm Sport is a compact and feature-rich smartwat…
    listed=$136.00  ensemble=$244.56  discount=$108.56
  Pricing: The Apple iPhone 13 128GB Smartphone is a powerful and versatile devic…
    listed=$194.00  ensemble=$717.00  discount=$523.00
  Pricing: The Anker 15W MagSafe Qi2 Wireless Charger is a convenient and efficie…
    listed=$15.00  ensemble=$23.88  discount=$8.88
  Pricing: The Soundcore by Anker P20i True Wireless Earbuds are a high-quality a…
    listed=$20.00  ensemble=$42.21  discount=$22.21
  Pricing: The Beats Studio Buds are a premium pair of wireless earbuds, designed…
    listed=$59.00  ensemble=$165.54  discount=$106.54

Done pricing all deals.


In [8]:
def _bar(v, available=True):
    if not available or v == 0:
        return "<td style='color:#aaa'>n/a</td>"
    return f"<td>${v:.2f}</td>"

header = """
<table style='border-collapse:collapse; font-family:monospace; width:100%'>
<thead>
<tr style='background:#222;color:#eee'>
  <th style='padding:6px 10px;text-align:left'>Deal</th>
  <th>Listed</th><th>Claude</th><th>Specialist</th><th>RAG</th><th>DNN</th>
  <th style='background:#445'>Ensemble</th><th style='background:#363'>Discount</th>
</tr>
</thead><tbody>
"""
rows = ""
for i, r in enumerate(results):
    bg = "#1a1a2e" if i % 2 == 0 else "#16213e"
    e  = r["estimates"]
    disc_color = "#4caf50" if r["discount"] > 50 else ("#ffc107" if r["discount"] > 0 else "#f44336")
    rows += f"""<tr style='background:{bg}'>
  <td style='padding:6px 10px;max-width:320px'>{r["deal"].product_description[:120]}…</td>
  <td>${r["deal"].price:.2f}</td>
  {_bar(e["claude"])}
  {_bar(e["specialist"], specialist_agent._available)}
  {_bar(e["rag"])}
  {_bar(e["dnn"], dnn_agent._available)}
  <td style='font-weight:bold'>${e["ensemble"]:.2f}</td>
  <td style='color:{disc_color};font-weight:bold'>${r["discount"]:.2f}</td>
</tr>"""

html_table = header + rows + "</tbody></table>"
display(HTML(html_table))


Deal,Listed,Claude,Specialist,RAG,DNN,Ensemble,Discount
"The Apple iPhone 13 128GB Smartphone is a powerful and versatile device, equipped with the Apple A15 Bionic chip and a s…",$194.00,$699.00,$680.00,$799.00,n/a,$717.00,$523.00
The Apple Watch SE 2 40mm Sport is a compact and feature-rich smartwatch designed for fitness enthusiasts and individual…,$136.00,$249.00,$199.00,$279.00,n/a,$244.56,$108.56
"The Beats Studio Buds are a premium pair of wireless earbuds, designed to provide a high-quality audio experience. With …",$59.00,$149.99,$220.00,$149.95,n/a,$165.54,$106.54
"The Soundcore by Anker P20i True Wireless Earbuds are a high-quality audio solution, designed to provide an immersive li…",$20.00,$39.99,$40.00,$49.99,n/a,$42.21,$22.21
"The Anker 15W MagSafe Qi2 Wireless Charger is a convenient and efficient charging solution, designed to provide fast and…",$15.00,$24.99,$25.00,$19.99,n/a,$23.88,$8.88


In [9]:
best = results[0]
deal = best["deal"]

print(f"\n🏆  Best deal found:\n")
print(f"  Description : {deal.product_description}")
print(f"  Listed price: ${deal.price:.2f}")
print(f"  Ensemble est: ${best['estimates']['ensemble']:.2f}")
print(f"  Discount    : ${best['discount']:.2f}")
print(f"  URL         : {deal.url}\n")

print("Crafting alert message with Claude (via MessagingAgent)…\n")
from agents.messaging_agent import MessagingAgent
messenger = MessagingAgent()
alert_text = messenger.craft_message(
    description=deal.product_description,
    deal_price=deal.price,
    estimated_true_value=best["estimates"]["ensemble"],
)
display(Markdown(f"### Deal Alert\n\n{alert_text}"))



🏆  Best deal found:

  Description : The Apple iPhone 13 128GB Smartphone is a powerful and versatile device, equipped with the Apple A15 Bionic chip and a stunning 6.1-inch AMOLED display. This smartphone offers an exceptional user experience, with advanced features such as high-quality camera capabilities, fast charging, and a long-lasting battery. Additionally, it features model MLML3LL/A and is designed to provide seamless performance and efficiency.
  Listed price: $194.00
  Ensemble est: $717.00
  Discount    : $523.00
  URL         : https://www.dealnews.com/products/Apple/Apple-iPhone-13-128-GB-Smartphone/277922.html?iref=rss-c142

Crafting alert message with Claude (via MessagingAgent)…



### Deal Alert

🚨 MASSIVE iPhone 13 Deal Alert! Get the Apple iPhone 13 128GB with A15 Bionic chip and stunning 6.1" AMOLED display for just $194 – that's a whopping 73% OFF the $717 retail value! This won't last long – grab this incredible steal on a powerful, feature-packed smartphone now!

## Key Improvements Over the Course Baseline

| Dimension | Course Baseline | This Exercise |
|-----------|----------------|---------------|
| Frontier LLM | GPT-5.1 (OpenAI) | Claude Sonnet 4.5 (Anthropic) |
| Ensemble members | Frontier + Specialist + DNN | + RAG Few-Shot (Week 7) |
| Preprocessor | Ollama (local) | Groq llama-3.1-8b-instant (cloud) |
| Scanner model | gpt-5-mini (OpenAI, quota-limited) | Groq llama-3.3-70b JSON mode (free tier) |
| Alert delivery | Pushover (needs token) | Rich notebook output |
| DNN weights absent | Crash | Graceful skip + renormalise weights |

### What We Learned
- **Claude vs GPT on RAG pricing:** Anthropic's Claude Sonnet proved a solid drop-in for frontier pricing — same Chroma RAG pipeline, different LLM endpoint.
- **Ensemble diversity helps:** Adding the Week-7 RAG Few-Shot agent as a 4th member reduces variance because it uses a completely different retrieval strategy (in-memory cosine) from the Chroma-backed FrontierAgent.
- **Graceful degradation is essential in agentic systems:** Hard failures in one agent (missing weights, cold Modal start) shouldn't crash the whole pipeline — renormalising weights at runtime keeps the ensemble working.
